# EmerGen-Brain — Free Colab/Kaggle Training (MobileNetV2)
Free T4 friendly: ~7k images, 128px, batch 32, ~10-15 min for 12 epochs.
Works on Colab **and** Kaggle (both free).

In [ ]:
# 1) Get code — option A: upload this repo to Colab, or clone:
# !git clone <your-repo-url> emergen-brain
# %cd emergen-brain

# 1b) If you run this notebook standalone, create the src files from the repo first.
# This cell installs deps (CPU+GPUTorch auto-selected by Colab).!pip -q install torch torchvision pillow numpy scikit-learn matplotlib pyyaml tqdm kagglehub onnx opencv-python-headless

In [ ]:
# 2) Download public brain MRI set (~7k, 4 classes)
import kagglehub
from pathlib import Path
src = Path(kagglehub.dataset_download('masoudnickparvar/brain-tumor-mri-dataset'))
print(src); print(list(src.iterdir())[:10])

In [ ]:
# 3) Flatten Training/Testing into data/<class>/
import shutil
from pathlib import Path
src = Path(src); out = Path('data')
for split in ['Training','Testing','training','testing','train','test','']:
    base = src/split if split else src
    if not base.exists(): continue
    for cls in ['glioma','meningioma','notumor','pituitary']:
        for cand in [base/cls, base/cls.capitalize()]:
            if cand.is_dir():
                (out/cls).mkdir(parents=True, exist_ok=True)
                for f in cand.iterdir():
                    if f.is_file() and f.suffix.lower() in ('.jpg','.jpeg','.png'):
                        shutil.copy(f, out/cls/f.name)
print({p.name: len(list(p.glob('*'))) for p in out.iterdir() if p.is_dir()})

In [ ]:
# 4) Train (uses src/train.py + config.yaml from repo)
!python src/train.py --config config.yaml

In [ ]:
# 5) Quick eval + sample Grad-CAM
!python src/evaluate.py --checkpoint outputs/best.pt --data data

from pathlib import Path
from src.infer import predict
import matplotlib.pyplot as plt
from PIL import Image
sample = next((Path('data')/'notumor').glob('*'))
r = predict(str(sample))
print(r['verdict']); print(r['why'])
fig, ax = plt.subplots(1,2, figsize=(8,4))
ax[0].imshow(Image.open(sample)); ax[0].set_title('input'); ax[0].axis('off')
ax[1].imshow(r['heatmap']); ax[1].set_title('Grad-CAM'); ax[1].axis('off')
plt.show()

## 6) Download the model → run locally on CPU
Files → `outputs/best.pt` → Download. Put it in your local `outputs/` folder, then:
```
pip install -r requirements.txt
python src/infer.py path/to/mri.jpg --heatmap-out heat.png
python app.py  # opens http://127.0.0.1:7860
```
Kaggle note: same cells work — enable GPU T4 in Settings → Accelerator, add `!pip install` cell first.